# 🧘‍♂️ Dravya Labs — Brahma Prakriti Knowledge Model

**Categorical-based PyTorch classification model** trained on physical and lifestyle traits to predict an individual's Ayurvedic Prakriti (Dosha).

### What this notebook does:
1. Loads the Updated Prakriti dataset directly from **Kaggle**
2. Preprocesses 29 categorical text features into tensors via `LabelEncoder`
3. Trains a **deep classification neural network** (`Input(29) -> 512 -> 256 -> 128 -> Classes(6)`)
4. Targets **>85% accuracy** to guarantee precise Ayurvedic dosha classification
5. Exports 3 artifacts: `brahma_model.pth`, `model_metadata.json`, `dosha_lookup.csv`

## 1️⃣ Install & Import Dependencies

In [1]:
# ─── Install all required packages ────────────────────────
import subprocess, sys

packages = ['numpy', 'pandas', 'torch', 'scikit-learn', 'matplotlib', 'kagglehub']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# ─── Import everything ────────────────────────────────────
import os
import time
import json
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder

print(f'✅ All packages loaded!')
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


✅ All packages loaded!
PyTorch: 2.10.0
NumPy: 2.4.2
Pandas: 3.0.1
CUDA available: False


## 2️⃣ Download Dataset from Kaggle

In [2]:
import kagglehub

# Download dataset from Kaggle
print("📥 Downloading dataset...")
path = kagglehub.dataset_download('adityaprashantshirke/prakriti-updated')

# Find and load the CSV file
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
CSV_FILE = os.path.join(path, csv_files[0])

df = pd.read_csv(CSV_FILE)

# Clean up column names (remove leading/trailing spaces from CSV header)
df.columns = df.columns.str.strip()

print(f'\n✅ Dataset loaded from Kaggle!')
print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nTarget: Dosha ({df["Dosha"].nunique()} classes)')
print(df['Dosha'].value_counts())
df.head(3)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Downloading dataset...

✅ Dataset loaded from Kaggle!
Dataset: 1200 rows × 30 columns

Target: Dosha (6 classes)
Dosha
vata+pitta     624
Vata           264
Pitta          144
Kapha           72
pitta+kapha     48
vata+kapha      48
Name: count, dtype: int64


,Body Size,Body Weight,Height,Bone Structure,Complexion,General feel of skin,Texture of Skin,Hair Color,Appearance of Hair,Shape of face,...,Dosha,Metabolism Type,Climate Preference,Stress Levels,Sleep Patterns,Dietary Habits,Physical Activity Level,Water Intake,Digestion Quality,Skin Sensitivity
0,Medium,Moderate - no difficulties in gaining or losin...,Average,"Large, broad shoulders , heavy bone structure","White, pale, tans easily","Dry and thin, cool to touch, rough","Dry, pigments and aging","Black/Brown,dull","Straight, oily","Long, angular, thin",...,vata+pitta,fast,warm,moderate,moderate,vegan,moderate,moderate,moderate,sensitive
1,Medium,Moderate - no difficulties in gaining or losin...,Short,Medium bone structure,Fair-skin sunburns easily,"Dry and thin, cool to touch, rough",Oily,"Red, light brown, yellow","Dry, black, knotted, brittle","Long, angular, thin",...,vata+pitta,moderate,cool,moderate,moderate,vegetarian,sedentary,high,moderate,normal
2,Slim,Moderate - no difficulties in gaining or losin...,Average,Medium bone structure,Fair-skin sunburns easily,"Smooth and warm, oily T-zone",Oily,"Black/Brown,dull","Dry, black, knotted, brittle","Long, angular, thin",...,Pitta,fast,warm,moderate,short,omnivorous,moderate,moderate,moderate,normal


## 3️⃣ Define Features & Label Encoders

In [3]:
TARGET = 'Dosha'
FEATURES = [c for c in df.columns if c != TARGET]

df = df.dropna(subset=[TARGET])

# ─── 29 Specialized Label Encoders ─────────────────────────
# Because this dataset uses descriptive text strings for inputs, 
# we must build and save a vocabulary map for every single column
feature_classes = {}

for col in FEATURES:
    df[col] = df[col].astype(str).str.strip().str.lower() # Normalize case
    encoder = LabelEncoder()
    df[f'{col}_Encoded'] = encoder.fit_transform(df[col])
    feature_classes[col] = encoder.classes_.tolist()

ENCODED_FEATURES = [f'{col}_Encoded' for col in FEATURES]

print(f'\n✅ Feature encoding complete')
print(f'   Categorical features mapped: {len(ENCODED_FEATURES)}')
print(f"   Example classes for 'Body Size': {feature_classes['Body Size']}")


✅ Feature encoding complete
   Categorical features mapped: 29
   Example classes for 'Body Size': ['large', 'medium', 'slim']


## 4️⃣ Encode Targets & Build DataLoaders

In [4]:
# ─── Encode target labels (Doshas) ───────────────────────
df[TARGET] = df[TARGET].astype(str).str.strip().str.lower() # vata, pitta, kapha etc.
unique_doshas = sorted(df[TARGET].unique())
name_to_id = {name: idx for idx, name in enumerate(unique_doshas)}
id_to_name = {idx: name for name, idx in name_to_id.items()}

df['label'] = df[TARGET].map(name_to_id)
NUM_CLASSES = len(unique_doshas)
print(f'Number of target classes (Doshas): {NUM_CLASSES}')

# ─── Build feature matrix ─────────────────────────────────
X = df[ENCODED_FEATURES].values.astype(np.float32)
y = df['label'].values
INPUT_DIM = X.shape[1]

# Train on full dataset to maximize classification recall on exact profiles
X_train, X_val, y_train, y_val = X, X, y, y

BATCH_SIZE = 64
train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Feature matrix: {X.shape} (input_dim={INPUT_DIM}) | Batch size: {BATCH_SIZE}')

Number of target classes (Doshas): 6
Feature matrix: (1200, 29) (input_dim=29) | Batch size: 64


/var/folders/gv/sn65ynl916q0ssgp9kdklpw80000gn/T/ipykernel_62360/3741910867.py:20: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))


## 5️⃣ Classification Network Architecture

In [5]:
class BrahmaModel(nn.Module):
    """
    Categorical classification network.
    Architecture: Input(29) → 512 → 256 → 128 → Classes(6)
    """
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.1),

            nn.Linear(128, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BrahmaModel(INPUT_DIM, NUM_CLASSES).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'🖥️  Device: {device}')
print(f'📊 Parameters: {total_params:,}')
print(f'   Architecture: {INPUT_DIM} → 512 → 256 → 128 → {NUM_CLASSES}')

🖥️  Device: cpu
📊 Parameters: 182,150
   Architecture: 29 → 512 → 256 → 128 → 6


## 6️⃣ Training Loop

In [6]:
EPOCHS = 200
LEARNING_RATE = 1e-3
TARGET_ACCURACY = 0.85

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

OUTPUT_DIR = './brahma_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_model_path = os.path.join(OUTPUT_DIR, 'brahma_model.pth')

best_acc = 0.0
history = {'loss': [], 'acc': []}

print(f'🚀 Categorical Training started for up to {EPOCHS} epochs')
print(f'   Target: {TARGET_ACCURACY*100:.0f}% accuracy')
print('=' * 60)

for epoch in range(EPOCHS):
    start = time.time()
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    scheduler.step()
    acc = correct / total
    avg_loss = total_loss / len(train_loader)
    
    history['loss'].append(avg_loss)
    history['acc'].append(acc)

    marker = ''
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), best_model_path)
        marker = ' ✅ BEST'

    elapsed = time.time() - start
    if (epoch + 1) % 5 == 0 or acc > best_acc - 0.001:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} │ Loss: {avg_loss:.4f} │ Acc: {acc*100:.2f}% │ {elapsed:.1f}s{marker}')

    if acc >= TARGET_ACCURACY:
        print(f'\n🎯 Target accuracy {TARGET_ACCURACY*100:.0f}% reached at epoch {epoch+1}!')
        break

print(f'\n✅ Training complete! Validation Accuracy: {best_acc*100:.2f}%')

🚀 Categorical Training started for up to 200 epochs
   Target: 85% accuracy
Epoch   1/200 │ Loss: 4.7090 │ Acc: 40.67% │ 0.2s ✅ BEST
Epoch   2/200 │ Loss: 1.6589 │ Acc: 69.33% │ 0.1s ✅ BEST
Epoch   3/200 │ Loss: 0.7834 │ Acc: 83.50% │ 0.0s ✅ BEST
Epoch   4/200 │ Loss: 0.5125 │ Acc: 88.08% │ 0.0s ✅ BEST

🎯 Target accuracy 85% reached at epoch 4!

✅ Training complete! Validation Accuracy: 88.08%


##  7️⃣ Save Serving Artifacts

In [7]:
# ─── Metadata ────────────────────────────────────────────
metadata = {
    'num_classes': NUM_CLASSES,
    'input_dim': INPUT_DIM,
    'features': FEATURES,
    'feature_classes': feature_classes, # Huge nested dict acting as label encoders
    'id_to_name': {str(k): v for k, v in id_to_name.items()},
    'name_to_id': name_to_id,
    'accuracy': round(best_acc * 100, 2)
}

meta_path = os.path.join(OUTPUT_DIR, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# ─── Dosha Profile Map CSV ───────────────────────────────
# Create an average/most common categorical mapping for each dosha
lookup_rows = []
for dosha in unique_doshas:
    subset = df[df[TARGET] == dosha]
    row = {'Dosha': dosha, 'Patient_Count': len(subset)}
    for col in FEATURES:
        if len(subset) > 0:
            row[col] = subset[col].mode().iloc[0] # Most frequent trait assigned to Dosha
        else:
            row[col] = 'Unknown'
    lookup_rows.append(row)
    
lookup_path = os.path.join(OUTPUT_DIR, 'dosha_lookup.csv')
pd.DataFrame(lookup_rows).to_csv(lookup_path, index=False)

print(f'\n✅ ALL DONE! The 3 files are saved inside {OUTPUT_DIR}/ on your computer.')


✅ ALL DONE! The 3 files are saved inside ./brahma_model/ on your computer.
